## 1. Установка google disk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

domain_path = '/content/drive/MyDrive/Colab Notebooks/graphRAG'
os.chdir(domain_path)
print(os.listdir())

Mounted at /content/drive
['RAG_example02.ipynb', 'prompts', 'domains', 'src']


In [ ]:
import os

domain_name = "HPotter"
domain_path = f"./domains/{domain_name}"

# Проверяем, существует ли папка, если нет — создаем
os.makedirs(domain_path, exist_ok=True)

fileslist = os.listdir(domain_path)
fileslist

['questions.xlsx', 'docs', 'markedocs', 'gpt-oss-120b.xlsx', 'RAG_HP.ipynb']

## 2. Готовим LLM-агента

In [ ]:
import requests
import json

from typing import Optional, List, Dict
from openai import OpenAI
import json


class YandexGPTAgentClass:

    def __init__(self, config_dict: dict):
        """
        Initializes the LLMAgent with a configuration dictionary.
        Expected config_dict keys: 'base_URL', 'api_key', 'base_model', 'model_config', 'cloud_folder_id'.
        """
        self.base_url = config_dict.get('base_URL')
        self.api_key = config_dict.get('api_key')
        self.model_name = config_dict.get('model_name')
        self.model_config = config_dict.get('model_config', {})
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key)
        # этот параметр характерен именно для вызова YaGPT моделей
        self.cloud_folder_id = config_dict.get('cloud_folder_id') # Добавлен cloud_folder_id
        if not self.cloud_folder_id:
            raise ValueError("YandexGPTAgent is not fully configured. Missing cloud_folder_id.")


    def invoke(self, messages: List[Dict[str, str]], **kwargs) -> str:
        """
        Runs a request to the Yandex GPT API using the OpenAI client,
        adding Yandex-specific headers (X-Folder-ID).

        Args:
            messages (List[Dict[str, str]]): A list of message dictionaries, each with 'role' and 'text' keys.
            **kwargs: Additional keyword arguments specific to the LLM API call (e.g., temperature).

        Returns:
            str: The response text from the LLM.

        Raises:
            ValueError: If the agent is not fully configured.
            Exception: For general errors during API invocation.
        """
        if not self.base_url or not self.api_key or not self.cloud_folder_id:
            raise ValueError("YandexGPTAgent is not fully configured. Missing base_url, api_key, or cloud_folder_id.")

        completion_options = {
            "temperature": self.model_config.get("temperature", 0.6),
            "max_tokens": self.model_config.get("max_tokens", 25000)
        }
        for key in ['temperature', 'max_tokens', 'seed']: # Allow kwargs to override
            if key in kwargs:
                completion_options[key] = kwargs[key]

        # extra_headers = {"X-Folder-ID": self.cloud_folder_id}

        try:
            chat_completion = self.client.chat.completions.create(
                model=f"gpt://{self.cloud_folder_id}/{self.model_name}",
                messages=messages,
                # extra_headers=extra_headers,
                # response_format=kwargs.get("response_format", {"type": "json_object"}), # Allow overriding response_format
                **completion_options
            )
            llm_output_str = chat_completion.choices[0].message.content
            return llm_output_str

        except Exception as e:
            raise Exception(f"Произошла ошибка при вызове YandexGPTAgent: {e}")

    def set_model(self, model_name: str, model_config: Optional[dict] = None):
        """
        Changes the base model name and/or updates the model parameters.

        Args:
            model_name (str): The new name of the model to use.
            model_config (Optional[dict]): A dictionary of new model parameters to update.
                                        If a key exists, its value will be updated; otherwise, it will be added.
        """
        if model_name:
            self.model_name = model_name
        if model_config is not None:
            # Ensure model_config exists in self.config before updating
            if self.model_config is None:
                self.model_config = {}
            self.model_config.update(model_config)

In [ ]:
YANDEX_API_KEY = ""  # yandex API
os.environ['YANDEX_API_KEY'] = YANDEX_API_KEY

# задаем параметры облачных моделей yandex AI-Studio
YANDEX_CLOUD_FOLDER = ""
os.environ['YANDEX_CLOUD_FOLDER'] = YANDEX_CLOUD_FOLDER

In [ ]:
YANDEX_BASE_URL = "https://ai.api.cloud.yandex.net/v1"  # OpenAI-совместимый базовый URL для YandexGPT API

YA_LLM_config = {
    "config_name": "DOCKER_LLM_config", # название конфигурации
    'base_URL': YANDEX_BASE_URL,
    "api_key": os.getenv('YANDEX_API_KEY'),
    "model_name": "yandexgpt-5.1/latest", # Название используемой модели по умолчанию
    "cloud_folder_id": os.getenv('YANDEX_CLOUD_FOLDER'), # Идентификатор каталога для X-Folder-ID
    "model_config": {"max_tokens": 30000}, # параметры модели по умолчанию
}

yandex_agent = YandexGPTAgentClass(YA_LLM_config)
print("YandexGPTAgent инстанциирован успешно.")

YandexGPTAgent инстанциирован успешно.


## Выбираем модель LLM и тестим агента на простом запросе

In [ ]:
# выберем модель LLM
model_name = "deepseek-v4-flash/latest"
model_name = "qwen3-235b-a22b-fp8/latest"
model_name = "gpt-oss-120b/latest"

model_config = {"max_tokens": 30000}
yandex_agent.set_model(model_name, model_config)

In [ ]:
# тестим агента

dialog_prompt = [
    {"role": "system", "content": "You are a helpful assistant for knowledge base creation."},
    {"role": "user", "content": "Кто написал книгу про гарри Поттера ?"}
]

try:
    response_text = yandex_agent.invoke(dialog_prompt)
    print("YandexGPTAgent Ответ:")
    print(response_text)
except Exception as e:
    print(f"Произошла ошибка при вызове YandexGPTAgent: {e}")

YandexGPTAgent Ответ:
{
  "answer": "Книгу о Гарри Поттере написала Джоан К. Роулинг (J.K. Rowling)."
}


## Прогоняем ответы на вопросы до RAG

In [ ]:
import re
import pandas as pd


# Функция для удаления лишних символов
def clean_response(text):
    # Убираем все маркеры типа [/] и любые специальные символы
    clean_text = re.sub(r"\[\/?\w+\]", "", text)  # Убираем [SYS], [USER], и другие токены
    clean_text = re.sub(r"<\|[^>]*\|>", "", clean_text)  # Убираем токены вроде <|start_header_id|>
    return clean_text.strip()  # Убираем пробелы в начале и конце

In [ ]:
import os

question_file_path = os.path.join(domain_path, 'questions.xlsx')


# Чтение Excel-файла
df = pd.read_excel(question_file_path)

curr_model = model_name[model_name.index("/")]
# Добавление новых колонок для ответов модели
df[curr_model] = ''
promt = "Ты эксперт вселенной Гарри Поттера из романа Джоан Роулинг 'Гарри Поттер и философский камень'. Отвечай по-русски и строго по сюжету книги. Не придумывай. "
# Прогон вопросов через модель и запись ответов в df
for index, row in df.iterrows():
    question = promt + row['Вопрос']
    response = yandex_agent.invoke([{"role": "user", "content": question}])
    print(response)
    clean_answer = clean_response(response)
    df.at[index, curr_model] = clean_answer


{ "answer": "В «Гарри Поттере и философском камне» мантия (домашний плащ) Гриффиндора описывается как ярко‑красного (алого) цвета с золотыми полосами и деталями. Это соответствует официальным цветам факультета — алый и золотой." }
{ "ответ": "В книге «Гарри Поттер и философский камень» говорится, что палочка Гарри сделана из остролиста (holly) длиной одиннадцать дюймов и имеет сердцевину из пера феникса. Этот перо принадлежало Фоуксу — фениксу Альбуса Дамблдора, и оно также использовалось в палочке Волдеморта." }
{ 
  "answer": "Самый популярный вид спорта среди волшебников — квиддич. В «Гарри Поттере и философском камне» он уже упоминается как «самый популярный спорт в волшебном мире», играется на метлах и имеет большую популярность во всех школах магии, в том числе и в Хогвартсе." 
}
{ "director": "Альбус Дамблдор" }
{  
  "answer": "Профессор Северус Снегг (Снейп) преподаёт в Хогвартсе предмет «Зельеварение» (англ. Potions) и одновременно является главой факультета Слизерин."  
}
{ 

In [ ]:
answer = """[
{ "answer": "Мантия Гриффиндорского дома в книге описана как красная (бордово‑красного оттенка) с золотыми элементами, соответствующими цветам дома – красному и золотому." },
{ "answer": "Палочка Гарри Поттера сделана из ольхи (holly) и имеет сердцевину из пера феникса — того же самого феникса Фоукса, принадлежащего Альбусу Дамблдору." },
{ "answer": "Самым популярным видом спорта среди волшебников, описанным в романе «Гарри Поттер и философский камень», является квиддич." },
{
  "answer": "Директора Хогвартса зовут Альбус Дамблдор."
},
{ "answer": "Профессор Снейп (Снегг) преподаёт предмет «Зельеварение»." },
{ "answer": "У Гарри в первой книге «Гарри Поттер и философский камень» используется метла модели **Nimbus 2000**. Она была подарена ему профессором Макгонагалл после того, как он был назначен поисковиком команды Гриффиндора." },
{ "answer": "Incendio", "explanation": "В книге «Гарри Поттер и философский камень» упоминается заклинание Incendio, которое используется для создания огня." },
{ "answer": "Лев" },
{ "answer": "Родителей Гарри Поттера убил Лорд Волдеморт (Том Марволо Риддл)." },
{ "spell": "Wingardium Leviosa", "description_ru": "В книге «Гарри Поттер и философский камень» для левитации предметов используется заклинание Вингардиум Левиоса (англ. Wingardium Leviosa)." },
{ "answer": "Джинни Уизли" },
{ "best_friends": ["Рон Уизли", "Гермиона Грэнджер"] },
{ "spell": "Lumos", "description": "Заклинание «Люмос» (лат. lumos) используется для создания света, например, при чтении в темных помещениях. Для его гашения применяется заклинание «Нокс»."},
{  "address": "4 Privet Drive, Little Whinging, Surrey" },
{
  "answer": "В книге «Гарри Поттер и философский камень» мистер (Верг) Дурсль упоминается как директор фирмы «Grunnings», которая производит сверла."
},
{
  "answer": "В книге «Гарри Поттер и философский камень» профессор Минерва МакГонагалл является анимагом и может превращаться в кота. В первой главе она появляется в виде большого пятнистого (рыжего) кота, сидящего на стене, а позже раскрывает, что это она сама.",
  "source": "Гарри Поттер и философский камень, Дж. К. Роулинг, глава «Ожерелье Дамблдора» (англ. «The Philosopher's Stone», глава 1).",
  "note": "Ответ основан строго на оригинальном сюжете книги без добавления вымышленных деталей."
},
{ "answer": "В Хогвартсе четыре факультета (дома): Гриффиндор, Хаффлпафф, Когтевран и Слизерин." },
{ "answer": "В книге «Гарри Поттер и философский камень» говорится, что Гарри отправлялся в Хогвартс с платформы 9 ¾ на вокзале Кингс‑Кросс в Лондоне." },
{ "answer": "В «Гарри Поттере и философском камне» в Запретном лесу был убит единорог. Именно его кровь использовал Волдеморт (через Квира) для восстановления своих сил." },
{},
{
  "vault_number": "713"
},
{ "name": "Тревор", "species": "жаба (жаба‑жаба, в оригинале — жаба‑жаба, в книге упоминается как жаба‑жаба)" },
{
  "character": "Сеймус Финниган",
  "attempted_transformation": "вино",
  "book": "Гарри Поттер и философский камень",
  "chapter": "9 – Полуночный поединок",
  "quote": "— Попробую превратить воду в вино! — воскликнул Сеймус, но заклинание вышло не так, как он рассчитывал, и в котелке произошёл небольшой взрыв."
},
{
  "answer": "Кошка Аргуса Филча называется Миссис Норрис."
},
{ "spell": "Wingardium Leviosa", "character": "Ron Weasley", "scene": "Бой с горным троллем в ванной комнате («Гарри Поттер и философский камень»)"},
{ "answer": "Энчант шахматной доски, которая служила одной из защитных преград к Философскому камню, наложила профессор Минерва МакГонагал." },
{ "answer": "В книгах Джоан Роулинг такие люди называют **маглами** (англ. Muggles)." },
{ "answer": "Собаку Хагрида зовут Фанг." },
{ "answer": "Профессор Минерва Макгонагалл преподает предмет «Трансфигурация»." },
{ "full_name": "Сэр Николас де Мимси‑Порпингтон", "nickname_ru": "Почти безголовый Ник", "nickname_en": "Nearly Headless Nick" },
{
  "answer": "В книге «Гарри Поттер и философский камень» у Гермионы нет кота. Её кот — Крукшэнкс — появляется только в более поздних книгах (начиная с «Гарри Поттер и узник Азкабана»)."
},
{
  "answer": "В романе «Гарри Поттер и философский камень» у Гермионы нет кота, поэтому о цвете его шерсти в этой книге речи нет.",
  "source": "Текст «Гарри Поттер и философский камень» (J.K. Rowling) – в нём нет упоминания о коте Гермионы."
},
{
  "answer": "Самый большой мяч, используемый в квиддиче, — это квоффл (Quaffle). Он примерно по размеру футбольного мяча и служит основной мячом, который пытаются забросить в кольца соперника."
},
{
  "answer": "В «Гарри Поттере и философском камне» кентавра, которого Гарри встречает в Запретном лесу, Джоан Роулинг не назвала. В книге он упоминается лишь как «кентавр», без имени."
},
{
  "answer": "Лучшим другом Даддли (Даддли Дурслей) в «Гарри Поттер и философский камень» называют Пирса Полкиса (Piers Polkiss)."
},
{
  "answer": "Внутри самого романа «Гарри Поттер и философский камень» ни один персонаж не назван автором книги. События описываются от лица рассказчика, но в тексте книги нет упоминания, кто написал эту историю. (В реальном мире, конечно, автором является Дж. К. Роулинг, но об этом в сюжете книги не говорится.)"
},
{ "answer": "Гарри Поттер" },
{ "answer": "Хедвига" },
{ "answer": "Гарри учится на факультете Гриффиндор." },
{ "best_friend": "Рон Уизли" },
{ "answer": "Гермиона Грейнджер" },
{
  "answer": "Директора школы Хогвартс зовут Альбус Дамблдор."
},
{ "enemy": "Волдеморт (Том Марволо Реддл)" },
{ "sport": "Квиддич", "position": "ловец", "house": "Гриффиндор" },
{ "answer": "Учителем зельеварения в Хогвартсе в книге «Гарри Поттер и философский камень» является Северус Снегг." },
{
  "answer": "Мать Гарри зовут Лили Поттер (в девичестве — Лили Эванс).",
  "language": "ru"
},
{ "father_name": "Джеймс Поттер" },
{ "answer": "Газета, которой пользуется большинство волшебников, называется «Ежедневный пророк»." },
{
  "answer": "Чарли Уизли"
},
{ "answer": "Билл Уизли" },
{
  "answer": "Джинни Уизли"
},
{ "answer": "Профессор Квиррелл (Квиррелл)" },
{ "answer": "Зеркало Эриса (англ. Mirror of Erised)" },
{
  "answer": "Школа называется Хогвартс — Школа чародейства и волшебства Хогвартс."
},
{ "owner": "Мистер Оливандер", "shop": "Оливандер и сын" },
{ "answer": "Запретный лес" },
{ "answer": "Профессор Минерва МакГонагалл" },
{
  "answer": "Дадли Дурслей"
},
{ "answer": "Вернон Дурсл" },
{ "answer": "Тётя Петунья Дурсли" },
{ "answer": "Сестру Лили Поттер зовут Петунья Эванс (в дальнейшем — Петунья Дурсль)." },
{ "answer": "В книге «Гарри Поттер и философский камень» философский камень охранял трехглавый пёс по имени Флаффи (Fluffy). Он был привязан к двери подземного хранилища, где находился камень, и мог быть усмирён только волшебной мелодией, сыгранной на волшебной флейте." },
{
  "answer": "Гарри играл на позиции Искателя (Seeker) в квиддиче."
},
{ "enemy": "Драко Малфой" },
{ "answer": "Минерва МакГонагал" },
{ "answer": "В книгах Джоан Роулинг такие люди называют **маглами** (англ. Muggle) — людьми, не обладающими магическими способностями и не происходящими из волшебных семей." },
{
  "answer": "В книге «Гарри Поттер и философский камень» экзамен SOV (в оригинале O.W.L. — Ordinary Wizarding Level) вовсе не упоминается. О W.L.‑экзамены вводятся только в пятом учебном году, когда ученики Хогвартса готовятся к сдаче этих обязательных испытаний в конце учебного года (обычно в июне). Поэтому в первой книге студенты ещё не сдают O.W.L.; они лишь проходят обычные уроки и периодические проверочные задания, но официального экзамена SOV/О.W.L. не происходит.",
  "source": "«Гарри Поттер и философский камень», Дж. К. Роулинг, 1997 г.",
  "notes": "Если подразумевался экзамен O.W.L., то он проводится в пятом классе, а не в первом, как описано в романе."
},
{
  "answer": "У Гарри Поттера день рождения — 31 июля (1980 года).",
  "source": "Книга «Гарри Поттер и философский камень», глава «Письмо» (упоминается, что Дурсли отмечают его день рождения 31 июля)."
},
{ "answer": "Собака Рубеуса Хагрида зовут Фанг." },
{"role":"assistant","content":"В квиддиче команда состоит из семи игроков: три ловца (чейзера), два битера, один голкипер и один искатель."},
{
  "character": "Чарли Уизли",
  "canon_source": [
    "«Гарри Поттер и Кубок огня» (глава 23)",
    "«Гарри Поттер и Орден Феникса» (глава 21)"
  ],
  "post_graduation_activity": "После окончания Хогвартса Чарли Уизли уехал работать с драконами. Он переехал в Румынию, где помогает в разведении и уходе за этими магическими существами, работая в одном из драконьих заповедников."
},
{
  "team": "Гриффиндор",
  "captain": "Оливер Вуд (вратарь)",
  "seeker": "Гарри Поттер",
  "chasers": [
    "Ангелина Джонсон",
    "Кэти Белл",
    "Алисия Спиннет"
  ],
  "beaters": [
    "Фред Уизли",
    "Джордж Уизли"
  ]
},
{
  "answer": "В книге «Гарри Поттер и философский камень» конкретного возраста Николаса Фламеля в момент, когда он решил (или согласился) уничтожить философский камень, не назвали. Однако в диалоге Дамблдора с Гарри (глава 17) сказано, что Фламель «… уже более шестисот лет живёт», а в «Гарри Поттер и Дары Смерти» уточняется, что в 1991 году ему 665 лет. Следовательно, когда в 1991 году Фламель согласился отдать камень и позволил его уничтожить, ему было около 665 лет.",
  "source": "Гарри Поттер и философский камень, глава 17; Гарри Поттер и Дары Смерти, эпилог"
},
{ "answer": "Последняя защита — это Зеркало Эризма (Зеркало, показывающее самое заветное желание сердца). Именно перед этим зеркалом Гарри сталкивается в последнем испытании, после чего через него попадает к камню." },
{
  "response": "В «Гарри Поттере и философском камне» нет персонажа, существа или предмета с именем «Пушка». В книге упоминаются такие существа, как трёхголовый пёс Флаффи, тролль в ванной, единорог в Запретном лесу, дракон‑яйцо с Норбертом и др., но никакой «Пушки» нет. Поэтому в рамках самого сюжета книги нельзя дать совет, как её успокоить, потому что её просто не существует в этом произведении.",
  "language": "ru",
  "adherence_to_plot": true,
  "no_invention": true
},
{"role":"assistant","content":"Распределением учеников по факультетам в Хогвартсе занималась **Распределяющая шляпа** (Sorting Hat). При первом совете в Большом зале Дамблдор надевал её на голову каждому новому ученику, и шляпа, опираясь на её магическое знание и мысли ученика, объявляла, к какому из четырёх факультетов (Гриффиндор, Слизерин, Когтевран или Пуффендуй) он будет зачислен. Всё это описано в «Гарри Поттере и философском камне» без каких‑либо отклонений от сюжета."},
{ "answer": "Профессор Квиррель" },
{
  "answer": "В книге «Гарри Поттер и философский камень» «дьявольские силки» (Devil’s Snare) — это волшебное растение, которое обвивает и душит того, кто в него попадает. Оно боится яркого света и тепла (огня). Именно этим они и пытаются его победить: Гермиона советует «свети фонариком» и «подожги спичку», а также говорит, что растение ослабнет, если перестать сопротивляться и расслабиться. Всё это описано в главе «Тайный проход», когда тройка школьников выходит из подземного коридора.",
  "source": "Гарри Поттер и философский камень, глава «Тайный проход», диалог Гермионы о дьявольских силках"
},
{
  "answer": "Гарри заменял ладью (rook) в волшебных шахматах."
},
{
  "answer": "В «Гарри Поттере и философском камне» банк Гринготтс полностью обслуживают гоблины. В книге мы видим, как гоблин‑кассир принимает у Хагрида и Гарри их деньги, открывает их ячейку № 711 и объясняет правила работы банка. Кроме того, гоблины охраняют все ячейки, в том числе и ту, где хранился философский камень, а также помогают Хагриду с доставкой золота. Других существ‑работников в Гринготтсе в этом томе не упоминается; всё управление и обслуживание банка принадлежит исключительно гоблинам."
},
{ "answer": "В волшебном мире Джоан Роулинг один галлеон равен семнадцати серебряным сиклям." },
{
  "answer": "Хозяйкой больничного крыла в Хогвартсе является Мадам Помфри (Madam Pomfrey)."
},
{
  "answer": "В «Гарри Поттере и философском камне» нет ни одной сцены, где Альбус Дамблдор получает или пробует карамельку Берти Боттс. В тексте упоминаются конфеты и сладости (шоколадные лягушки, леденцы, сливки и т.п.), но конкретно «Бёрти Боттс» Дамблдору не достаётся, и, следовательно, вкус такой конфетки в книге не назван.",
  "source": "Книга «Гарри Поттер и философский камень», Джоан Роулинг, оригинальный английский текст (и русский перевод).",
  "note": "Ответ основывается строго на содержании первой книги; любые дополнительные детали из последующих томов или внешних источников здесь не учитываются."
}
]
"""

In [ ]:
os.listdir(domain_path)

['questions.xlsx', 'docs', 'markedocs', 'RAG_HP.ipynb']

In [ ]:
# Сохранение DataFrame обратно в Excel
curr_model = model_name[:model_name.index("/")]
filename = curr_model+'.xlsx'
filepath = os.path.join(domain_path, filename)
print(filepath)
df.to_excel(filepath, index=False)

./domains/HPotter/gpt-oss-120b.xlsx


## 4. Строим RAG

1. Загрузим исходный текст, который будем использовать для поиска контекста для ответа на запрос

In [ ]:
filename = "docs/HarryPotter.txt"
filepath = os.path.join(domain_path, filename)
print(filepath)
with open(filepath, 'r', encoding='utf-8') as file:
    text = file.read()

print(f"Загружен текст из файла {filename} длиной {len(text)} символов")

./domains/HPotter/docs/HarryPotter.txt
Загружен текст из файла docs/HarryPotter.txt длиной 552188 символов


2. Формируем массив фрагментов: делим текст на чанки

In [ ]:
from pydantic import Field, BaseModel
from typing import Optional, List, Union, Dict, Any, Annotated

class ChunkClass(BaseModel):
    """ извлеченный чанк """
    serial_num: int = Field(..., description="сквозной номер чанка в данной части документа")
    content: str = Field(..., description="содержимое чанка")


class ExtractChunkResult(BaseModel):
    """Результат извлечения чанков"""
    chunks: List[ChunkClass] = Field(..., description="извлеченные чанки")

In [ ]:
chunk_size = 1000
chunk_overlap = 100
chunks_list = []
serial_num = 1

start = 0
while start < len(text):
    end = min(start + chunk_size, len(text))
    content = text[start:end]
    chunks_list.append(ChunkClass(serial_num=serial_num, content=content))
    serial_num += 1
    if end == len(text):
        break
    start += chunk_size - chunk_overlap

extract_result = ExtractChunkResult(chunks=chunks_list)
print(f"Создано чанков: {len(extract_result.chunks)}")

Создано чанков: 614


In [ ]:
import json
import os

# Путь для сохранения JSON-файла с чанками
chunks_json_path = os.path.join(domain_path, 'chunks.json')

# Сериализуем объект extract_result в формат JSON с поддержкой UTF-8
with open(chunks_json_path, 'w', encoding='utf-8') as f:
    json.dump(extract_result.model_dump(), f, ensure_ascii=False, indent=2)

print(f"Чанки успешно сохранены в файл: {chunks_json_path}")

Чанки успешно сохранены в файл: ./domains/HPotter/chunks.json


In [ ]:
def create_text_qa_template():
    """
    Создает шаблон для запросов Q&A.
    """
    chat_text_qa_msgs = [
        ChatMessage(
            role=MessageRole.SYSTEM,
            content=(
                "Ты эксперт вселенной Гарри Поттера из романа Джоан Роулинг 'Гарри Поттер и философский камень'. Отвечай по-русски и строго по контексту. Не придумывай."
            ),
        ),
        ChatMessage(
            role=MessageRole.USER,
            content=(
                "Контекстная информация приведена ниже.\n"
                "---------------------\n"
                "{context_str}\n"
                "---------------------\n"
                "Теперь ответь на вопрос с учетом контекста: {query_str}\n"
            ),
        ),
    ]
    text_qa_template = ChatPromptTemplate(chat_text_qa_msgs)
    return text_qa_template

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Подготовка текстов чанков для поиска
chunk_texts = [chunk.content for chunk in extract_result.chunks]

# Инициализация и обучение TF-IDF векторного представления
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

def retrieve_context(query: str, top_k: int = 3) -> str:
    """
    Находит top_k наиболее релевантных чанков для заданного вопроса.
    """
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(similarity)[-top_k:][::-1]

    retrieved_chunks = [chunk_texts[idx] for idx in top_indices]
    return "\n---\n".join(retrieved_chunks)

def make_rag(query: str) -> str:
    """
    Основная функция RAG: извлекает контекст и генерирует ответ с помощью LLM.
    """
    context = retrieve_context(query, top_k=3)

    messages = [
        {
            "role": "system",
            "content": "Ты эксперт вселенной Гарри Поттера из романа Джоан Роулинг 'Гарри Поттер и философский камень'. Отвечай по-русски и строго по контексту. Не придумывай."
        },
        {
            "role": "user",
            "content": f"Контекстная информация приведена ниже.\n---------------------\n{context}\n---------------------\nТеперь ответь на вопрос с учетом контекста: {query}"
        }
    ]

    try:
        return yandex_agent.invoke(messages)
    except Exception as e:
        return f"Ошибка RAG: {e}"

# Проверим работу на тестовом вопросе
test_query = "Какая модель метлы была у Гарри Поттера в первой книге?"
print("Тестовый ответ RAG:")
print(make_rag(test_query))

Тестовый ответ RAG:
В первой книге у Гарри Поттера была метла модели **«Нимбус‑2000»** (Nimbus 2000).


## 5. Теперь прогоним вопросы через RAG

In [ ]:
curr_model = model_name[:model_name.index("/")]
curr_model, model_name

('gpt-oss-120b', 'gpt-oss-120b/latest')

In [ ]:
import pandas as pd


curr_model = model_name[:model_name.index("/")]
filename = curr_model+'.xlsx'
filepath = os.path.join(domain_path, filename)
print(filepath)

# Чтение Excel-файла
df = pd.read_excel(filepath)
# Добавление новой колонки для ответов RAG
df['RAG_'+ curr_model] = ''

# Прогон вопросов через RAG и запись ответов в DataFrame
for index, row in df.iterrows():
    question = row['Вопрос']
    rag_answer = make_rag(question)
    df.at[index, 'RAG_'+ curr_model] = rag_answer

# Сохранение DataFrame обратно в Excel
out_filename = 'rag_'+curr_model+'.xlsx'
out_filepath = os.path.join(domain_path, out_filename)
print(out_filepath)
df.to_excel(out_filepath, index=False)

./domains/HPotter/gpt-oss-120b.xlsx
./domains/HPotter/rag_gpt-oss-120b.xlsx


## Оценим работу нейросетей без RAG м с RAG, используя LLM-as-Judge

In [ ]:
df.columns

Index(['Вопрос', 'Ответ', 'gpt-oss-120b', 'gpt-oss-human', 'RAG_gpt-oss-120b'], dtype='object')

In [ ]:
import re
import time

# Установим для нашего агента модель deepseek-v4-flash
yandex_agent.set_model("deepseek-v4-flash/latest", {"max_tokens": 3000})

def evaluate_answer(question: str, reference: str, candidate_answer: str) -> float:
    """
    Запрашивает у LLM-судьи оценку ответа от 0 до 1.
    """
    if not str(candidate_answer).strip():
        return 0.0

    prompt = [
        {
            "role": "system",
            "content": "Ты беспристрастный судья. Оцени, содержит ли ответ кандидата правильную информацию по сравнению с эталонным ответом. Верни строго одно число от 0.0 (полностью неверно) до 1.0 (полностью верно и содержит верную информацию). Не пиши никаких пояснений, только число."
        },
        {
            "role": "user",
            "content": f"Вопрос: {question}\nЭталонный ответ: {reference}\nОтвет кандидата: {candidate_answer}\n\nТвоя оценка (число от 0.0 до 1.0):"
        }
    ]

    try:
        response = yandex_agent.invoke(prompt)
        # Ищем число с плавающей точкой в ответе
        match = re.search(r"\b(0\.\d+|1\.0|0|1)\b", response.strip())
        if match:
            return float(match.group(1))
        return 0.0
    except Exception as e:
        print(f"Ошибка при оценке: {e}")
        return 0.0

# Создаем новые колонки для оценок
df['Score_gpt-oss-120b'] = 0.0
df['Score_RAG_gpt-oss-120b'] = 0.0

print("Начинаем оценку ответов...")
for index, row in df.iterrows():
    q = row['Вопрос']
    ref = row['Ответ']
    ans_no_rag = row['gpt-oss-120b']
    ans_rag = row['RAG_gpt-oss-120b']

    score_no_rag = evaluate_answer(q, ref, ans_no_rag)
    score_rag = evaluate_answer(q, ref, ans_rag)

    df.at[index, 'Score_gpt-oss-120b'] = score_no_rag
    df.at[index, 'Score_RAG_gpt-oss-120b'] = score_rag

    if index % 10 == 0 or index == len(df) - 1:
        print(f"Обработано строк: {index + 1}/{len(df)}")
    time.sleep(0.1) # Небольшая пауза во избежание лимитов API

# Сохраняем обновленный датафрейм
evaluated_filepath = os.path.join(domain_path, 'evaluated_rag_gpt-oss-120b.xlsx')
df.to_excel(evaluated_filepath, index=False)
print(f"Оценка завершена. Результаты сохранены в: {evaluated_filepath}")
print("Средняя оценка БЕЗ RAG:", df['Score_gpt-oss-120b'].mean())
print("Средняя оценка С RAG:", df['Score_RAG_gpt-oss-120b'].mean())

Начинаем оценку ответов...
Обработано строк: 1/83
Обработано строк: 11/83
Обработано строк: 21/83
Обработано строк: 31/83
Обработано строк: 41/83
Обработано строк: 51/83
Обработано строк: 61/83
Обработано строк: 71/83
Обработано строк: 81/83
Обработано строк: 83/83
Оценка завершена. Результаты сохранены в: ./domains/HPotter/evaluated_rag_gpt-oss-120b.xlsx
Средняя оценка БЕЗ RAG: 0.8626506024096385
Средняя оценка С RAG: 0.7301204819277107


## RAG на основе embedder-моделей
----
Реализуем RAG с использованием эмбеддеров. Чтобы модель токенайзера-мбеддера можно было бы задавать/менять.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

class EmbedderRAG:
    def __init__(self, model_name: str = "intfloat/multilingual-e5-large"):
        """
        Инициализирует эмбеддер моделью из Hugging Face.
        Для моделей семейства E5 рекомендуется использовать префиксы "query: " и "passage: ".
        """
        print(f"Загрузка модели эмбеддингов {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model_name = model_name
        self.chunks = []
        self.chunk_embeddings = None

    def fit(self, chunks: list):
        """
        Сохраняет чанки текста и вычисляет их векторные представления.
        """
        self.chunks = chunks
        print(f"Вычисление эмбеддингов для {len(chunks)} фрагментов текста...")

        # Для моделей E5 важно добавлять префикс "passage: " перед кодированием документов
        if "e5" in self.model_name.lower():
            passages = [f"passage: {chunk}" for chunk in chunks]
        else:
            passages = chunks

        self.chunk_embeddings = self.model.encode(passages, show_progress_bar=True, convert_to_numpy=True)
        print("Эмбеддинги успешно вычислены.")

    def retrieve(self, query: str, top_k: int = 3) -> str:
        """
        Выполняет семантический поиск по эмбеддингам и возвращает объединенный контекст.
        """
        if self.chunk_embeddings is None:
            raise ValueError("Сначала необходимо вызвать метод .fit(chunks)")

        # Для моделей E5 важно добавлять префикс "query: " перед кодированием поискового запроса
        if "e5" in self.model_name.lower():
            processed_query = f"query: {query}"
        else:
            processed_query = query

        query_embedding = self.model.encode([processed_query], convert_to_numpy=True)

        # Считаем косинусное сходство
        similarities = cosine_similarity(query_embedding, self.chunk_embeddings).flatten()
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        retrieved_chunks = [self.chunks[idx] for idx in top_indices]
        return "\n---\n".join(retrieved_chunks)

# 1. Инициализируем наш новый RAG с моделью multilingual-e5-large
emb_rag = EmbedderRAG(model_name="intfloat/multilingual-e5-large")

# 2. Передаем тексты чанков в эмбеддер
chunk_texts = [chunk.content for chunk in extract_result.chunks]
emb_rag.fit(chunk_texts)

# 3. Функция генерации ответа на основе извлеченного контекста через эмбеддер
def make_emb_rag(query: str, top_k: int = 3) -> str:
    context = emb_rag.retrieve(query, top_k=top_k)

    messages = [
        {
            "role": "system",
            "content": "Ты эксперт вселенной Гарри Поттера из романа Джоан Роулинг 'Гарри Поттер и философский камень'. Отвечай по-русски и строго по контексту. Не придумывай."
        },
        {
            "role": "user",
            "content": f"Контекстная информация приведена ниже.\n---------------------\n{context}\n---------------------\nТеперь ответь на вопрос с учетом контекста: {query}"
        }
    ]

    try:
        return yandex_agent.invoke(messages)
    except Exception as e:
        return f"Ошибка RAG: {e}"

# 4. Проверка работы семантического поиска
test_query = "Какая модель метлы была у Гарри Поттера в первой книге?"
print("\nТестовый ответ RAG на основе эмбеддингов:")
print(make_emb_rag(test_query))

Загрузка модели эмбеддингов intfloat/multilingual-e5-large...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Вычисление эмбеддингов для 614 фрагментов текста...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Эмбеддинги успешно вычислены.

Тестовый ответ RAG на основе эмбеддингов:
Модель метлы Гарри Поттера — «Нимбус-2000».


In [ ]:
import time
import os

# 1. Возвращаем активную модель генератора gpt-oss-120b для чистоты эксперимента
yandex_agent.set_model("gpt-oss-120b/latest", {"max_tokens": 30000})

print("Генерация ответов с использованием EmbedderRAG (E5-large)... ")
# Добавляем колонку для ответов E5 RAG
df['RAG_e5_gpt-oss-120b'] = ''

for index, row in df.iterrows():
    question = row['Вопрос']
    # Используем созданную ранее функцию make_emb_rag, работающую на E5
    rag_answer = make_emb_rag(question, top_k=3)
    df.at[index, 'RAG_e5_gpt-oss-120b'] = rag_answer
    if (index + 1) % 10 == 0 or index == len(df) - 1:
        print(f"Сгенерировано ответов: {index + 1}/{len(df)}")
    time.sleep(0.05)

# 2. Переключаемся на судью deepseek-v4-flash для оценки полученных результатов
yandex_agent.set_model("deepseek-v4-flash/latest", {"max_tokens": 3000})

print("\nОценка ответов нового RAG-эмбеддера с помощью LLM-as-Judge...")
df['Score_RAG_e5_gpt-oss-120b'] = 0.0

for index, row in df.iterrows():
    q = row['Вопрос']
    ref = row['Ответ']
    ans_e5_rag = row['RAG_e5_gpt-oss-120b']

    score_e5_rag = evaluate_answer(q, ref, ans_e5_rag)
    df.at[index, 'Score_RAG_e5_gpt-oss-120b'] = score_e5_rag

    if (index + 1) % 10 == 0 or index == len(df) - 1:
        print(f"Оценено ответов: {index + 1}/{len(df)}")
    time.sleep(0.05)

# 3. Сохранение результатов и вывод статистики
final_filepath = os.path.join(domain_path, 'evaluated_rag_e5_gpt-oss-120b.xlsx')
df.to_excel(final_filepath, index=False)

print(f"\nОценка завершена! Результаты успешно сохранены в: {final_filepath}")
print("--------------------------------------------")
print("Средняя оценка БЕЗ RAG:             ", df['Score_gpt-oss-120b'].mean())
print("Средняя оценка с TF-IDF RAG:        ", df['Score_RAG_gpt-oss-120b'].mean())
print("Средняя оценка с Embedder RAG (E5):  ", df['Score_RAG_e5_gpt-oss-120b'].mean())


Генерация ответов с использованием EmbedderRAG (E5-large)... 
Сгенерировано ответов: 10/83
Сгенерировано ответов: 20/83
Сгенерировано ответов: 30/83
Сгенерировано ответов: 40/83
Сгенерировано ответов: 50/83
Сгенерировано ответов: 60/83
Сгенерировано ответов: 70/83
Сгенерировано ответов: 80/83
Сгенерировано ответов: 83/83

Оценка ответов нового RAG-эмбеддера с помощью LLM-as-Judge...
Оценено ответов: 10/83
Оценено ответов: 20/83
Оценено ответов: 30/83
Оценено ответов: 40/83
Оценено ответов: 50/83
Оценено ответов: 60/83
Оценено ответов: 70/83
Оценено ответов: 80/83
Оценено ответов: 83/83

Оценка завершена! Результаты успешно сохранены в: ./domains/HPotter/evaluated_rag_e5_gpt-oss-120b.xlsx
--------------------------------------------
Средняя оценка БЕЗ RAG:              0.8626506024096385
Средняя оценка с TF-IDF RAG:         0.7301204819277107
Средняя оценка с Embedder RAG (E5):   0.8048192771084337
